In [ ]:
# ============================================================
# QUETTA OCR — English + Urdu Handwriting Recognition
# Works in: Google Colab + GitHub Pages
# Folders: input_images/ → output_text/
# ============================================================

# ── STEP 1: Install dependencies ──────────────────────────
!pip install easyocr opencv-python-headless matplotlib pillow -q

# ── STEP 2: Imports ───────────────────────────────────────
import cv2
import numpy as np
import easyocr
import os
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

# ── STEP 3: Setup folders ─────────────────────────────────
INPUT_FOLDER  = "input_images"
OUTPUT_FOLDER = "output_text"
os.makedirs(INPUT_FOLDER,  exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"✅ Folders ready: {INPUT_FOLDER}/ → {OUTPUT_FOLDER}/")

# ── STEP 4: Initialize EasyOCR (English + Urdu) ───────────
print("⏳ Loading OCR models (first time takes 2-3 min)...")
reader = easyocr.Reader(['en', 'ur'], gpu=False)
print("✅ OCR models loaded!")

# ── STEP 5: Image Preprocessing ───────────────────────────
def preprocess_image(image_bgr):
    """
    Advanced preprocessing for handwriting:
    - Grayscale conversion
    - Bilateral denoise (preserves edges)
    - Adaptive threshold (handles uneven lighting)
    - Deskew (fix tilted text)
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    # Denoise
    gray = cv2.bilateralFilter(gray, d=9, sigmaColor=75, sigmaSpace=75)

    # Adaptive threshold — best for handwriting
    thr = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 11
    )

    # Deskew
    coords = np.column_stack(np.where(thr < 128))
    if len(coords) > 100:
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        if abs(angle) < 30:
            (h, w) = thr.shape
            M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
            thr = cv2.warpAffine(thr, M, (w, h),
                                 flags=cv2.INTER_CUBIC,
                                 borderMode=cv2.BORDER_REPLICATE)
    return thr


# ── STEP 6: OCR on single image ───────────────────────────
def run_ocr(image_path):
    """
    Run OCR on one image.
    Returns: list of (text, confidence, bbox)
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        print(f"  ⚠️ Cannot read: {image_path}")
        return []

    processed = preprocess_image(img_bgr)

    # EasyOCR reads preprocessed image
    results = reader.readtext(processed, detail=1, paragraph=False)
    return results


# ── STEP 7: Save results ──────────────────────────────────
def save_results(image_name, results, output_folder):
    """
    Saves:
    - .txt  → plain extracted text
    - .json → detailed results with confidence scores
    """
    base = Path(image_name).stem
    txt_path  = Path(output_folder) / f"{base}.txt"
    json_path = Path(output_folder) / f"{base}.json"

    lines = []
    json_data = []
    for (bbox, text, conf) in results:
        lines.append(text)
        json_data.append({
            "text": text,
            "confidence": round(float(conf), 4),
            "bbox": [list(map(int, pt)) for pt in bbox]
        })

    txt_path.write_text("\n".join(lines), encoding="utf-8")
    json_path.write_text(json.dumps(json_data, ensure_ascii=False, indent=2), encoding="utf-8")
    return txt_path, json_path


# ── STEP 8: Visualize result ──────────────────────────────
def visualize(image_path, results):
    """Draw bounding boxes and text on image"""
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for (bbox, text, conf) in results:
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(img_rgb, [pts], True, (201, 168, 76), 2)
        x, y = pts[0]
        cv2.putText(img_rgb, f"{text} ({conf:.0%})",
                    (x, max(y - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (201, 168, 76), 1)

    plt.figure(figsize=(14, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"OCR Result — {Path(image_path).name}", fontsize=13)
    plt.tight_layout()
    plt.show()


# ── STEP 9: Process ALL images in input_images/ ───────────
def process_all():
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    images = [f for f in Path(INPUT_FOLDER).iterdir() if f.suffix.lower() in exts]

    if not images:
        print("⚠️ No images found in input_images/ — upload some first!")
        return

    print(f"\n📂 Found {len(images)} image(s). Starting OCR...\n")
    summary = []

    for img_path in images:
        print(f"🔍 Processing: {img_path.name}")
        results = run_ocr(img_path)

        if not results:
            print(f"  ❌ No text detected.\n")
            continue

        txt_path, json_path = save_results(img_path.name, results, OUTPUT_FOLDER)
        extracted = " | ".join([r[1] for r in results])
        avg_conf  = sum(r[2] for r in results) / len(results)

        print(f"  ✅ Text: {extracted[:80]}...")
        print(f"  📊 Confidence: {avg_conf:.0%}")
        print(f"  💾 Saved: {txt_path.name}, {json_path.name}\n")

        summary.append({
            "file": img_path.name,
            "words_found": len(results),
            "avg_confidence": f"{avg_conf:.0%}",
            "output_txt": str(txt_path)
        })

        visualize(img_path, results)

    # Save summary
    summary_path = Path(OUTPUT_FOLDER) / "summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"📋 Summary saved: {summary_path}")
    print(f"\n✅ Done! All results in '{OUTPUT_FOLDER}/' folder.")


# ── STEP 10: Upload images manually (Colab) ───────────────
def upload_images():
    """Upload images directly from your computer to Colab"""
    print("📤 Select image files to upload...")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = Path(INPUT_FOLDER) / fname
        dest.write_bytes(data)
        print(f"  ✅ Saved: {dest}")
    print(f"\n✅ {len(uploaded)} file(s) uploaded to {INPUT_FOLDER}/")


# ── STEP 11: Download all output files ────────────────────
def download_outputs():
    """Zip and download all output files"""
    import zipfile
    zip_path = "ocr_output.zip"
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in Path(OUTPUT_FOLDER).iterdir():
            zf.write(f, f.name)
    files.download(zip_path)
    print(f"✅ Downloaded: {zip_path}")


# ══════════════════════════════════════════════════════════
# ▶ RUN — Choose one:
# ══════════════════════════════════════════════════════════

# Option A: Upload images from your computer
upload_images()

# Option B: Process all images already in input_images/
process_all()

# Option C: Download results after processing
# download_outputs()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 10.1 MB/s eta 0:00:00


✅ Folders ready: input_images/ → output_text/
⏳ Loading OCR models (first time takes 2-3 min)...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ OCR models loaded!
📤 Select image files to upload...
